# Corrected analysis: Iris control

## Question and estimand

Summarize sepal and petal measurement distributions by species in the 150-row sample. Report means and sample-mean intervals, then show pairwise mean differences descriptively. No causal or population-wide estimand is identified by the supplied sampling information.

## Estimates and comparisons

Each species has 50 records. The corrected table reports the mean with a 95% t interval and a seeded percentile-bootstrap interval for each of the four measurements. Pairwise mean differences use bootstrap percentile intervals; all 12 are pointwise, not simultaneous familywise intervals. Cohen's d is descriptive and does not turn this sample into a validated classifier.

Observed petal ranges separate setosa from the other two species. Versicolor and virginica have overlapping petal ranges. The sample means of virginica exceed versicolor on each of the four measurements; this is a statement about group means, not every individual flower.

## Limitations

Intervals quantify uncertainty in means under row-resampling and independence assumptions. They do not predict individual measurements. Three exact duplicate rows remain in the dataset; identical rounded values alone are insufficient evidence that they are duplicate specimens. Sampling provenance is limited, and these descriptive differences do not establish biological mechanisms or causation.

The code below prints only aggregate estimates and observed ranges.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'audit_tools.py').is_file())
sys.path.insert(0, str(project_root / 'src'))
from iris_audit import audit_iris_control

audit = audit_iris_control()
summary = pd.DataFrame(audit['summary_statistics'])
summary['t 95% CI (cm)'] = summary['t_ci_95_cm'].map(lambda ci: f'{ci[0]:.3f} to {ci[1]:.3f}')
summary['bootstrap 95% CI (cm)'] = summary['bootstrap_ci_95_cm'].map(lambda ci: f'{ci[0]:.3f} to {ci[1]:.3f}')
display(summary[['species', 'measurement', 'n', 'mean_cm', 'sd_cm', 't 95% CI (cm)', 'bootstrap 95% CI (cm)']].round(3))

comparisons = pd.DataFrame(audit['pairwise_comparisons'])
comparisons['contrast'] = comparisons['first_species'] + ' - ' + comparisons['second_species']
comparisons['pointwise bootstrap 95% CI (cm)'] = comparisons['bootstrap_ci_95_cm'].map(lambda ci: f'{ci[0]:+.3f} to {ci[1]:+.3f}')
display(comparisons[['measurement', 'contrast', 'mean_difference_cm', 'pointwise bootstrap 95% CI (cm)', 'cohens_d']].round(3))

ranges = pd.DataFrame(audit['observed_range_overlaps'])
display(ranges[['measurement', 'first_species', 'first_range_cm', 'second_species', 'second_range_cm', 'observed_ranges_overlap']])

print(audit['interpretation'])

,species,measurement,n,mean_cm,sd_cm,t 95% CI (cm),bootstrap 95% CI (cm)
0,Iris-setosa,sepal length,50,5.006,0.352,4.906 to 5.106,4.908 to 5.100
1,Iris-setosa,sepal width,50,3.418,0.381,3.310 to 3.526,3.314 to 3.522
2,Iris-setosa,petal length,50,1.464,0.174,1.415 to 1.513,1.416 to 1.512
3,Iris-setosa,petal width,50,0.244,0.107,0.214 to 0.274,0.216 to 0.274
4,Iris-versicolor,sepal length,50,5.936,0.516,5.789 to 6.083,5.798 to 6.078
5,Iris-versicolor,sepal width,50,2.770,0.314,2.681 to 2.859,2.680 to 2.854
6,Iris-versicolor,petal length,50,4.260,0.470,4.126 to 4.394,4.124 to 4.386
7,Iris-versicolor,petal width,50,1.326,0.198,1.270 to 1.382,1.274 to 1.382
8,Iris-virginica,sepal length,50,6.588,0.636,6.407 to 6.769,6.416 to 6.766
9,Iris-virginica,sepal width,50,2.974,0.322,2.882 to 3.066,2.886 to 3.064


,measurement,contrast,mean_difference_cm,pointwise bootstrap 95% CI (cm),cohens_d
0,sepal length,Iris-setosa - Iris-versicolor,-0.930,-1.098 to -0.758,-2.104
1,sepal length,Iris-setosa - Iris-virginica,-1.582,-1.788 to -1.382,-3.077
2,sepal length,Iris-versicolor - Iris-virginica,-0.652,-0.876 to -0.428,-1.126
3,sepal width,Iris-setosa - Iris-versicolor,0.648,+0.516 to +0.780,1.857
4,sepal width,Iris-setosa - Iris-virginica,0.444,+0.306 to +0.578,1.258
5,sepal width,Iris-versicolor - Iris-virginica,-0.204,-0.328 to -0.078,-0.641
6,petal length,Iris-setosa - Iris-versicolor,-2.796,-2.930 to -2.654,-7.894
7,petal length,Iris-setosa - Iris-virginica,-4.088,-4.248 to -3.936,-9.993
8,petal length,Iris-versicolor - Iris-virginica,-1.292,-1.490 to -1.096,-2.521
9,petal width,Iris-setosa - Iris-versicolor,-1.082,-1.144 to -1.020,-6.802


,measurement,first_species,first_range_cm,second_species,second_range_cm,observed_ranges_overlap
0,sepal length,Iris-setosa,"[4.3, 5.8]",Iris-versicolor,"[4.9, 7.0]",True
1,sepal length,Iris-setosa,"[4.3, 5.8]",Iris-virginica,"[4.9, 7.9]",True
2,sepal length,Iris-versicolor,"[4.9, 7.0]",Iris-virginica,"[4.9, 7.9]",True
3,sepal width,Iris-setosa,"[2.3, 4.4]",Iris-versicolor,"[2.0, 3.4]",True
4,sepal width,Iris-setosa,"[2.3, 4.4]",Iris-virginica,"[2.2, 3.8]",True
5,sepal width,Iris-versicolor,"[2.0, 3.4]",Iris-virginica,"[2.2, 3.8]",True
6,petal length,Iris-setosa,"[1.0, 1.9]",Iris-versicolor,"[3.0, 5.1]",False
7,petal length,Iris-setosa,"[1.0, 1.9]",Iris-virginica,"[4.5, 6.9]",False
8,petal length,Iris-versicolor,"[3.0, 5.1]",Iris-virginica,"[4.5, 6.9]",True
9,petal width,Iris-setosa,"[0.1, 0.6]",Iris-versicolor,"[1.0, 1.8]",False


{'pairwise_intervals_are_pointwise': True, 'pairwise_interval_family_size': 12, 'inference_scope': 'Descriptive for the observed sample; population generalization requires a defensible sampling design.', 'duplicate_row_caveat': 'Exact duplicate rows are retained. Identical rounded measurements alone do not establish duplicate specimens.'}
